# Semi-supervised approach for survival modelling
This notebook presents the development and evaluation of the proposed semi-supervised approach for survival modelling. The methodology consists of two main stages: a representation learning pre-training phase and a downstream survival fine-tuning phase.

In the first stage, a self-supervised contrastive learning framework is employed to learn representations from irregular multivariate time series. The approach follows the Time–Frequency Consistency (TFC) paradigm proposed by Zhang et al. (2022), which leverages both time-domain and frequency-domain views of the data. This design encourages the learned representations to remain consistent across different transformations and mitigates common issues in self-supervised learning, such as representation collapse and mismatch with downstream tasks.

To further align the learned representations with the survival prediction objective, the pretraining phase is weakly guided by survival information. This introduces a supervisory signal during representation learning, thereby framing the overall approach as semi-supervised.

In the second stage, the pretrained encoder is used to extract sequence representations, which are then combined with static vehicle specifications. Static features are incorporated using two strategies: (i) direct concatenation of one-hot encoded features, and (ii) dimensionality reduction through a single-layer multilayer perceptron (MLP) that compresses the static features into lower-dimensional representations (16 and 32 dimensions).

The combined representations are subsequently fine-tuned for survival prediction using a Cox-based model. Specifically, a two-layer MLP is used as a survival head to produce risk scores, and the model is trained using the Cox partial log-likelihood loss to optimize ranking performance.

Model selection is performed based on validation performance, and the best-performing configuration is evaluated on the test set. For transparency, the test performance of alternative model variants is also reported.

In addition to the concordance index (C-index), further evaluation metrics are considered to provide a more comprehensive assessment. Time-dependent AUC and Integrated Brier Score (IBS) are computed. Since the proposed model produces risk scores rather than explicit survival probabilities, the Breslow estimator is applied using the training data to estimate the baseline hazard function and derive survival probability curves. These curves are then used to compute the additional metrics.

Finally, the proposed approach is compared against traditional tabular survival models, including the Cox Proportional Hazards model and Random Survival Forests, using the same truncated dataset to ensure a fair comparison.

In [ ]:
# fix randomness
import torch
import numpy as np
import random

seed = 42

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# Set project paths and enable imports from src, models and data directories
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "sequences"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [SRC_DIR, DATA_DIR, MODELS_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

print(f"Notebook location: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Sequences directory: {DATA_DIR}")

In [ ]:
# check runtime GPU
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# load the truncated sequences

import pickle

with open(DATA_DIR / "train_sequences_truncated.pkl", "rb") as f:
    truncated_train_seq = pickle.load(f)

with open(DATA_DIR / "val_sequences_truncated.pkl", "rb") as f:
    truncated_val_seq = pickle.load(f)

with open(DATA_DIR / "test_sequences_truncated.pkl", "rb") as f:
    truncated_test_seq = pickle.load(f)
print(len(truncated_train_seq), len(truncated_val_seq))

In [ ]:
# define datasets
from surv_fin_tun_dataset import SurvivalSequenceDataset, survival_collate_fn
train_surv_dataset = SurvivalSequenceDataset(
    sequence_dict=truncated_train_seq,
    use_static_features=True,
)

val_surv_dataset = SurvivalSequenceDataset(
    sequence_dict=truncated_val_seq,
    use_static_features=True,
)

test_surv_dataset = SurvivalSequenceDataset(
    sequence_dict=truncated_test_seq,
    use_static_features=True,
)

print(len(train_surv_dataset), len(val_surv_dataset), len(test_surv_dataset))

In [ ]:
sample = train_surv_dataset[0]

print("Sample keys:", sample.keys())
print("Vehicle id:", sample["vehicle_id"])
print("Sequence length:", sample["sequence_length"])
print("Numerical shape:", sample["numerical"].shape)
print("Histogram shape:", sample["histogram"].shape)
print("Duration:", sample["duration"].item())
print("Event:", sample["event"].item())

if "readout_time" in sample:
    print("Readout time:", sample["readout_time"].item())

if "original_duration" in sample:
    print("Original duration:", sample["original_duration"].item())

In [ ]:
# make dataloaders
from torch.utils.data import DataLoader
# for the train loader, I would need to built a function that ensures
# each batch contains some event cases.

val_surv_loader = DataLoader(
    val_surv_dataset,
    batch_size=256,
    shuffle=False,
    collate_fn=survival_collate_fn,
    num_workers=0,
)

test_surv_loader = DataLoader(
    test_surv_dataset,
    batch_size=256,
    shuffle=False,
    collate_fn=survival_collate_fn,
    num_workers=0,
)

In [ ]:
# enforcing number of events in batch

from __future__ import annotations

import math
from typing import Iterator
import numpy as np
from torch.utils.data import Sampler


class EventAwareBatchSampler(Sampler[list[int]]):
    """
    Batch sampler that ensures each batch contains a minimum number of event samples.

    Parameters
    ----------
    events : list[int] | np.ndarray
        Binary event indicators aligned with dataset indexing, where 1 = event and 0 = censored.
    batch_size : int
        Total batch size.
    event_batch_size : int
        Number of event samples to include in each batch.
    drop_last : bool, default=False
        Whether to drop the final incomplete batch.
    random_seed : int | None, default=None
        Optional random seed for reproducibility.
    """

    def __init__(
        self,
        events: list[int] | np.ndarray,
        batch_size: int,
        event_batch_size: int,
        drop_last: bool = False,
        random_seed: int | None = None,
    ) -> None:
        if batch_size <= 0:
            raise ValueError("batch_size must be positive.")

        if event_batch_size <= 0:
            raise ValueError("event_batch_size must be positive.")

        if event_batch_size > batch_size:
            raise ValueError("event_batch_size cannot exceed batch_size.")

        self.events = np.asarray(events, dtype=np.int64)
        self.batch_size = batch_size
        self.event_batch_size = event_batch_size
        self.censored_batch_size = batch_size - event_batch_size
        self.drop_last = drop_last
        self.rng = np.random.default_rng(random_seed)

        self.event_indices = np.where(self.events == 1)[0]
        self.censored_indices = np.where(self.events == 0)[0]

        if len(self.event_indices) == 0:
            raise ValueError("No event samples found. Event-aware batching is not possible.")

        if self.censored_batch_size > 0 and len(self.censored_indices) == 0:
            raise ValueError("No censored samples found, but censored_batch_size > 0.")

    def __iter__(self) -> Iterator[list[int]]:
        # Shuffle event and censored pools independently each epoch
        event_indices = self.event_indices.copy()
        censored_indices = self.censored_indices.copy()

        self.rng.shuffle(event_indices)
        self.rng.shuffle(censored_indices)

        event_ptr = 0
        censored_ptr = 0

        num_event = len(event_indices)
        num_censored = len(censored_indices)

        while event_ptr < num_event:
            batch_indices = []

            # --- Event samples ---
            event_end = min(event_ptr + self.event_batch_size, num_event)
            event_batch = event_indices[event_ptr:event_end]
            batch_indices.extend(event_batch.tolist())
            event_ptr = event_end

            # If we could not fill the required event quota
            if len(event_batch) < self.event_batch_size:
                if self.drop_last:
                    break

            # --- Censored samples ---
            if self.censored_batch_size > 0:
                if censored_ptr + self.censored_batch_size <= num_censored:
                    censored_batch = censored_indices[
                        censored_ptr:censored_ptr + self.censored_batch_size
                    ]
                    censored_ptr += self.censored_batch_size
                else:
                    # If censored pool is exhausted, reshuffle and reuse
                    self.rng.shuffle(censored_indices)
                    censored_ptr = 0
                    censored_batch = censored_indices[
                        censored_ptr:censored_ptr + self.censored_batch_size
                    ]
                    censored_ptr += self.censored_batch_size

                batch_indices.extend(censored_batch.tolist())

            # Shuffle inside batch so events are not always first
            self.rng.shuffle(batch_indices)

            if len(batch_indices) < self.batch_size and self.drop_last:
                break

            yield batch_indices

    def __len__(self) -> int:
        if self.drop_last:
            return len(self.event_indices) // self.event_batch_size
        return math.ceil(len(self.event_indices) / self.event_batch_size)

In [ ]:
# build event vectors aligned with dataset indexing

train_events = [
    truncated_train_seq[vehicle_id]["event"]
    for vehicle_id in sorted(truncated_train_seq.keys())
]

In [ ]:
# create the sampler
batch_size = 128
event_batch_size = 32

train_batch_sampler = EventAwareBatchSampler(
    events=train_events,
    batch_size=batch_size,
    event_batch_size=event_batch_size,
    drop_last=False,
    random_seed=seed,
)

In [ ]:
# now make the event aware train loader
train_surv_loader = DataLoader(
    train_surv_dataset,
    batch_sampler=train_batch_sampler,
    collate_fn=survival_collate_fn,
    num_workers=0,
)

In [ ]:
# Import model modules
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from ssl_tfc_model import TFCSequenceModel, TFCSurvivalFineTuningModel, TFCSurvivalFineTuningWithStaticModel
from ssl_model import SSLSequenceModel, SurvivalFineTuningModel

In [ ]:
# class for self-supervised pre-training (ensuring time, frequency and consistency)
# across learned representations

import torch
import torch.nn.functional as F


"""
Losses and augmentations for TFC-style self-supervised pretraining.

This module provides:
- weak augmentations for the time branch
- weak augmentations for the frequency branch
- NT-Xent contrastive loss
- time-frequency consistency loss
- a wrapper that combines the losses into the TFC objective

Design principles
-----------------
- Augmentations are intentionally weak to avoid distorting the degradation
  structure of the operational trajectories.
- The time branch uses realistic perturbations in the sequence domain.
- The frequency branch uses light perturbations on the FFT magnitude space.
- The final loss combines:
    * time contrastive loss
    * frequency contrastive loss
    * time-frequency consistency loss
"""


def weak_time_augmentation(
    x: torch.Tensor,
    padding_mask: torch.Tensor,
    noise_std: float = 0.005,
    feature_dropout_prob: float = 0.02,
) -> torch.Tensor:
    """
    Apply weak augmentation to the time-domain sequence.

    Parameters
    ----------
    x : torch.Tensor
        Input tensor of shape (B, T, D).
    padding_mask : torch.Tensor
        Boolean mask of shape (B, T), where True indicates valid positions.
    noise_std : float, default=0.01
        Standard deviation of additive Gaussian noise.
    feature_dropout_prob : float, default=0.05
        Probability of dropping each feature value at valid positions.

    Returns
    -------
    torch.Tensor
        Augmented tensor of shape (B, T, D).
    """
    valid_mask = padding_mask.unsqueeze(-1).float()

    # Add small noise only on valid positions
    noise = torch.randn_like(x) * noise_std
    x_aug = x + noise * valid_mask

    # Apply small feature-wise dropout only on valid positions
    keep_mask = (
        torch.rand_like(x_aug) > feature_dropout_prob
    ).float()

    x_aug = x_aug * (keep_mask * valid_mask + (1.0 - valid_mask))
    return x_aug


def weak_frequency_augmentation(
    fft_mag: torch.Tensor,
    dropout_prob: float = 0.02,
    noise_std: float = 0.005,
) -> torch.Tensor:
    """
    Apply weak augmentation to the FFT magnitude representation.

    Parameters
    ----------
    fft_mag : torch.Tensor
        Frequency magnitude tensor of shape (B, T_f, D).
    dropout_prob : float, default=0.05
        Probability of dropping each value.
    noise_std : float, default=0.01
        Standard deviation of additive Gaussian noise.

    Returns
    -------
    torch.Tensor
        Augmented FFT magnitude tensor of shape (B, T_f, D).
    """
    noise = torch.randn_like(fft_mag) * noise_std
    fft_aug = fft_mag + noise

    keep_mask = (torch.rand_like(fft_aug) > dropout_prob).float()
    fft_aug = fft_aug * keep_mask

    return fft_aug


def nt_xent_loss(
    z1: torch.Tensor,
    z2: torch.Tensor,
    temperature: float = 0.2,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    Compute NT-Xent contrastive loss between two batches of projections.

    Parameters
    ----------
    z1 : torch.Tensor
        Projection tensor of shape (B, D).
    z2 : torch.Tensor
        Projection tensor of shape (B, D).
    temperature : float, default=0.2
        Temperature parameter.
    eps : float, default=1e-8
        Small constant for numerical stability.

    Returns
    -------
    torch.Tensor
        Scalar contrastive loss.
    """
    if z1.shape != z2.shape:
        raise ValueError("z1 and z2 must have the same shape.")

    batch_size = z1.size(0)
    if batch_size < 2:
        raise ValueError("NT-Xent requires batch_size >= 2.")

    z1 = F.normalize(z1, dim=1, eps=eps)
    z2 = F.normalize(z2, dim=1, eps=eps)

    representations = torch.cat([z1, z2], dim=0)  # (2B, D)
    similarity = torch.matmul(representations, representations.T) / temperature

    # Mask self-similarity
    diagonal_mask = torch.eye(2 * batch_size, device=z1.device, dtype=torch.bool)
    similarity = similarity.masked_fill(diagonal_mask, float("-inf"))

    # Positive pairs:
    # first half matches second half, second half matches first half
    positive_indices = torch.cat([
        torch.arange(batch_size, 2 * batch_size, device=z1.device),
        torch.arange(0, batch_size, device=z1.device),
    ])

    loss = F.cross_entropy(similarity, positive_indices)
    return loss


def time_frequency_consistency_loss(
    time_proj: torch.Tensor,
    freq_proj: torch.Tensor,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    Encourage time and frequency projections of the same sample to be close.

    Parameters
    ----------
    time_proj : torch.Tensor
        Time-branch projections of shape (B, D).
    freq_proj : torch.Tensor
        Frequency-branch projections of shape (B, D).
    eps : float, default=1e-8
        Small constant for numerical stability.

    Returns
    -------
    torch.Tensor
        Scalar consistency loss.
    """
    if time_proj.shape != freq_proj.shape:
        raise ValueError("time_proj and freq_proj must have the same shape.")

    time_proj = F.normalize(time_proj, dim=1, eps=eps)
    freq_proj = F.normalize(freq_proj, dim=1, eps=eps)

    # 1 - cosine similarity
    cosine_sim = F.cosine_similarity(time_proj, freq_proj, dim=1, eps=eps)
    loss = 1.0 - cosine_sim
    return loss.mean()


def tfc_total_loss(
    time_proj_1: torch.Tensor,
    time_proj_2: torch.Tensor,
    freq_proj_1: torch.Tensor,
    freq_proj_2: torch.Tensor,
    temperature: float = 0.2,
    lambda_tf: float = 0.5,
) -> dict[str, torch.Tensor]:
    """
    Compute the full TFC objective.

    Parameters
    ----------
    time_proj_1 : torch.Tensor
        Time-branch projection for augmented view 1, shape (B, D).
    time_proj_2 : torch.Tensor
        Time-branch projection for augmented view 2, shape (B, D).
    freq_proj_1 : torch.Tensor
        Frequency-branch projection for augmented view 1, shape (B, D).
    freq_proj_2 : torch.Tensor
        Frequency-branch projection for augmented view 2, shape (B, D).
    temperature : float, default=0.2
        Temperature parameter for NT-Xent.
    lambda_tf : float, default=0.5
        Weight controlling the balance between within-space contrastive losses
        and cross-space consistency loss.

    Returns
    -------
    dict[str, torch.Tensor]
        Dictionary containing:
        - "loss"
        - "loss_time"
        - "loss_freq"
        - "loss_consistency"
    """
    loss_time = nt_xent_loss(
        z1=time_proj_1,
        z2=time_proj_2,
        temperature=temperature,
    )

    loss_freq = nt_xent_loss(
        z1=freq_proj_1,
        z2=freq_proj_2,
        temperature=temperature,
    )

    loss_consistency_1 = time_frequency_consistency_loss(
        time_proj=time_proj_1,
        freq_proj=freq_proj_1,
    )

    loss_consistency_2 = time_frequency_consistency_loss(
        time_proj=time_proj_2,
        freq_proj=freq_proj_2,
    )

    loss_consistency = 0.5 * (loss_consistency_1 + loss_consistency_2)

    loss = lambda_tf * (loss_time + loss_freq) + (1.0 - lambda_tf) * loss_consistency

    return {
        "loss": loss,
        "loss_time": loss_time,
        "loss_freq": loss_freq,
        "loss_consistency": loss_consistency,
    }

In [ ]:
import torch
import torch.nn as nn
import numpy as np


class SurvivalGuidedTFCModel(nn.Module):
    def __init__(
        self,
        tfc_model: TFCSequenceModel,
        survival_hidden_dim: int = 64,
    ):
        super().__init__()

        self.tfc_model = tfc_model
        encoder_hidden_dim = tfc_model.time_encoder.hidden_dim

        self.survival_head = nn.Sequential(
            nn.Linear(encoder_hidden_dim, survival_hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(survival_hidden_dim, survival_hidden_dim),
            nn.ReLU(),
            nn.Linear(survival_hidden_dim, 1),
        )

    def forward(
        self,
        x_time,
        x_freq,
        time_gaps,
        padding_mask,
    ):
        outputs = self.tfc_model(
            x_time=x_time,
            x_freq=x_freq,
            time_gaps=time_gaps,
            padding_mask=padding_mask,
        )

        risk = self.survival_head(outputs["time_pooled"]).squeeze(-1)

        outputs["risk"] = risk
        return outputs

In [ ]:
# define the negative Cox partial log likelihood as the loss function

def cox_partial_log_likelihood(
    risk_scores: torch.Tensor,
    durations: torch.Tensor,
    events: torch.Tensor,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    Negative Cox partial log-likelihood.

    Parameters
    ----------
    risk_scores : torch.Tensor
        Predicted scalar risk scores of shape (B,).
    durations : torch.Tensor
        Observed durations of shape (B,).
    events : torch.Tensor
        Event indicators of shape (B,), where 1 = event and 0 = censored.
    eps : float, default=1e-8
        Small constant to avoid division by zero in edge cases.

    Returns
    -------
    torch.Tensor
        Scalar Cox loss.
    """
    if risk_scores.ndim != 1:
        raise ValueError("risk_scores must have shape (B,).")
    if durations.ndim != 1:
        raise ValueError("durations must have shape (B,).")
    if events.ndim != 1:
        raise ValueError("events must have shape (B,).")

    # Sort by descending duration
    order = torch.argsort(durations, descending=True)

    risk_scores = risk_scores[order]
    events = events[order]

    # log(sum_{j in R_i} exp(r_j)) using cumulative log-sum-exp
    log_risk_set_sum = torch.logcumsumexp(risk_scores, dim=0)

    # Contributions only from observed events
    event_mask = events == 1

    if event_mask.sum() == 0:
        # No uncensored event in batch
        return torch.tensor(0.0, device=risk_scores.device, requires_grad=True)

    observed_risk = risk_scores[event_mask]
    observed_log_risk_set_sum = log_risk_set_sum[event_mask]

    partial_log_lik = observed_risk - observed_log_risk_set_sum

    loss = -partial_log_lik.mean()
    return loss

In [ ]:
from sksurv.metrics import concordance_index_censored


def evaluate_survival_guided_tfc(
    model,
    data_loader,
    device,
):
    model.eval()

    total_loss = 0.0
    num_batches = 0

    all_risks = []
    all_durations = []
    all_events = []

    with torch.no_grad():
        for batch in data_loader:
            batch = {
                k: v.to(device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }

            x_time = torch.cat(
                [batch["numerical"], batch["histogram"]],
                dim=-1,
            )

            x_freq = torch.zeros_like(x_time)
            x_freq[:, 1:, :] = x_time[:, 1:, :] - x_time[:, :-1, :]

            out = model(
                x_time=x_time,
                x_freq=x_freq,
                time_gaps=batch["time_gaps"],
                padding_mask=batch["padding_mask"],
            )

            risk = out["risk"]

            if batch["event"].sum() > 0:
                loss = cox_partial_log_likelihood(
                    risk_scores=risk,
                    durations=batch["duration"],
                    events=batch["event"],
                )

                total_loss += loss.item()
                num_batches += 1

            all_risks.append(risk.detach().cpu())
            all_durations.append(batch["duration"].detach().cpu())
            all_events.append(batch["event"].detach().cpu())

    all_risks = torch.cat(all_risks).numpy()
    all_durations = torch.cat(all_durations).numpy()
    all_events = torch.cat(all_events).numpy().astype(bool)

    if all_events.sum() == 0:
        val_c_index = np.nan
    else:
        val_c_index = concordance_index_censored(
            all_events,
            all_durations,
            all_risks,
        )[0]

    avg_loss = total_loss / max(num_batches, 1)

    return avg_loss, val_c_index

In [ ]:
# training loop with TFC + Cox loss
def train_survival_guided_tfc(
    model,
    train_loader,
    val_loader,
    device,
    num_epochs=20,
    learning_rate=1e-4,
    weight_decay=0.0,
    temperature=0.2,
    lambda_tf=0.5,
    lambda_surv=0.3,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = {
        "train_total_loss": [],
        "train_tfc_loss": [],
        "train_cox_loss": [],
        "train_time_loss": [],
        "train_freq_loss": [],
        "train_consistency_loss": [],
        "val_loss": [],
        "val_c_index": [],
    }

    best_val_c_index = -np.inf
    best_state_dict = None
    best_tfc_state_dict = None

    for epoch in range(num_epochs):
        model.train()

        total_loss_sum = 0.0
        tfc_loss_sum = 0.0
        cox_loss_sum = 0.0
        time_loss_sum = 0.0
        freq_loss_sum = 0.0
        consistency_loss_sum = 0.0
        num_batches = 0
        skipped_batches = 0

        for batch in train_loader:
            batch = {
                k: v.to(device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }

            if batch["event"].sum() == 0:
                skipped_batches += 1
                continue

            x_time = torch.cat(
                [batch["numerical"], batch["histogram"]],
                dim=-1,
            )

            # For TFC design, x_freq is the differenced sequence.
            # Use the same x_time if you do not already store differenced input.
            # Better option is shown below.
            x_freq = torch.zeros_like(x_time)
            x_freq[:, 1:, :] = x_time[:, 1:, :] - x_time[:, :-1, :]

            x_time_1 = weak_time_augmentation(
                x=x_time,
                padding_mask=batch["padding_mask"],
            )

            x_time_2 = weak_time_augmentation(
                x=x_time,
                padding_mask=batch["padding_mask"],
            )

            x_freq_1 = weak_time_augmentation(
                x=x_freq,
                padding_mask=batch["padding_mask"],
            )

            x_freq_2 = weak_time_augmentation(
                x=x_freq,
                padding_mask=batch["padding_mask"],
            )

            out_1 = model(
                x_time=x_time_1,
                x_freq=x_freq_1,
                time_gaps=batch["time_gaps"],
                padding_mask=batch["padding_mask"],
            )

            out_2 = model(
                x_time=x_time_2,
                x_freq=x_freq_2,
                time_gaps=batch["time_gaps"],
                padding_mask=batch["padding_mask"],
            )

            # Risk from original non-augmented sequence
            out_original = model(
                x_time=x_time,
                x_freq=x_freq,
                time_gaps=batch["time_gaps"],
                padding_mask=batch["padding_mask"],
            )

            tfc_losses = tfc_total_loss(
                time_proj_1=out_1["time_proj"],
                time_proj_2=out_2["time_proj"],
                freq_proj_1=out_1["freq_proj"],
                freq_proj_2=out_2["freq_proj"],
                temperature=temperature,
                lambda_tf=lambda_tf,
            )

            cox_loss = cox_partial_log_likelihood(
                risk_scores=out_original["risk"],
                durations=batch["duration"],
                events=batch["event"],
            )

            loss = tfc_losses["loss"] + lambda_surv * cox_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss_sum += loss.item()
            tfc_loss_sum += tfc_losses["loss"].item()
            cox_loss_sum += cox_loss.item()
            time_loss_sum += tfc_losses["loss_time"].item()
            freq_loss_sum += tfc_losses["loss_freq"].item()
            consistency_loss_sum += tfc_losses["loss_consistency"].item()
            num_batches += 1

        avg_total_loss = total_loss_sum / max(num_batches, 1)
        avg_tfc_loss = tfc_loss_sum / max(num_batches, 1)
        avg_cox_loss = cox_loss_sum / max(num_batches, 1)

        # Evaluate using the survival-guided wrapper's risk head
        val_loss, val_c_index = evaluate_survival_guided_tfc(
            model=model,
            data_loader=val_loader,
            device=device,
        )

        history["train_total_loss"].append(avg_total_loss)
        history["train_tfc_loss"].append(avg_tfc_loss)
        history["train_cox_loss"].append(avg_cox_loss)
        history["train_time_loss"].append(time_loss_sum / max(num_batches, 1))
        history["train_freq_loss"].append(freq_loss_sum / max(num_batches, 1))
        history["train_consistency_loss"].append(
            consistency_loss_sum / max(num_batches, 1)
        )
        history["val_loss"].append(val_loss)
        history["val_c_index"].append(val_c_index)

        if not np.isnan(val_c_index) and val_c_index > best_val_c_index:
            best_val_c_index = val_c_index
            best_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            best_tfc_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.tfc_model.state_dict().items()
            }

        print(
            f"Epoch {epoch+1:02d} | "
            f"Total: {avg_total_loss:.4f} | "
            f"TFC: {avg_tfc_loss:.4f} | "
            f"Cox: {avg_cox_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val C-index: {val_c_index:.4f} | "
            f"Skipped: {skipped_batches}"
        )

    return {
        "history": history,
        "best_val_c_index": best_val_c_index,
        "best_state_dict": best_state_dict,
        "best_tfc_state_dict": best_tfc_state_dict,
    }

In [ ]:
# helper function for randomly initializing tfc_ssl model
# for survival guided pretraining
def build_survival_guided_tfc_model_from_scratch(
    time_input_dim: int = 105,
    freq_input_dim: int = 105,
    time_hidden_dim: int = 128,
    freq_hidden_dim: int = 128,
    projection_dim: int = 64,
    survival_hidden_dim: int = 64,
    device: torch.device = device,
):
    base_tfc = TFCSequenceModel(
        time_input_dim=time_input_dim,
        freq_input_dim=freq_input_dim,
        time_hidden_dim=time_hidden_dim,
        freq_hidden_dim=freq_hidden_dim,
        projection_dim=projection_dim,
    )

    model = SurvivalGuidedTFCModel(
        tfc_model=base_tfc,
        survival_hidden_dim=survival_hidden_dim,
    )

    return model.to(device)

In [ ]:
# seed for reproducibility, before building the model
torch.manual_seed(42)
np.random.seed(42)
# build the model
guided_model = build_survival_guided_tfc_model_from_scratch()
# pretrain ssl weakly guided with survival labels
guided_results = train_survival_guided_tfc(
    model=guided_model,
    train_loader=train_surv_loader,
    val_loader=val_surv_loader,
    device=device,
    num_epochs=40,
    learning_rate=1e-4,
    temperature=0.2,
    lambda_tf=0.5,
    lambda_surv=0.3,
)

In [ ]:
# save survival-guided pretrained model
save_path_tfc = MODELS_DIR / "survival_guided_tfc_results.pt"

torch.save({
    "model_state_dict": guided_model.state_dict(),
    "best_tfc_state_dict": guided_results["best_tfc_state_dict"],
    "history": guided_results["history"],
    "best_val_c_index": guided_results["best_val_c_index"],
    "config": {
        "time_input_dim": 105,
        "freq_input_dim": 105,
        "time_hidden_dim": 128,
        "freq_hidden_dim": 128,
        "projection_dim": 64,
        "survival_hidden_dim": 64,
    },
}, save_path_tfc)

In [ ]:
# fine tune the guided encoder
checkpoint = torch.load(save_path_tfc, map_location=device, weights_only=False)

guided_base_tfc = TFCSequenceModel(
    time_input_dim=checkpoint["config"]["time_input_dim"],
    freq_input_dim=checkpoint["config"]["freq_input_dim"],
    time_hidden_dim=checkpoint["config"]["time_hidden_dim"],
    freq_hidden_dim=checkpoint["config"]["freq_hidden_dim"],
    projection_dim=checkpoint["config"]["projection_dim"],
).to(device)

guided_base_tfc.load_state_dict(
    guided_results["best_tfc_state_dict"]
)

guided_survival_model = TFCSurvivalFineTuningModel(
    pretrained_tfc_model=guided_base_tfc,
    survival_hidden_dim=64,
    freeze_encoder=False,
).to(device)

In [ ]:
from sksurv.metrics import concordance_index_censored
import numpy as np


def concordance_index_from_risk(
    risk_scores: np.ndarray,
    durations: np.ndarray,
    events: np.ndarray,
) -> float:
    """
    Compute concordance index from predicted risk scores using sksurv.

    Parameters
    ----------
    risk_scores : np.ndarray
        Predicted risk scores of shape (N,). Higher means riskier.
    durations : np.ndarray
        Observed durations of shape (N,).
    events : np.ndarray
        Event indicators of shape (N,), where 1 = event and 0 = censored.

    Returns
    -------
    float
        Concordance index.
    """
    c_index = concordance_index_censored(
        event_indicator=events.astype(bool),
        event_time=durations,
        estimate=risk_scores,
    )[0]

    return float(c_index)

In [ ]:
# validation function

import numpy as np
import torch


def evaluate_survival_model(
    model: torch.nn.Module,
    data_loader,
    device: torch.device,
    model_type: str = "simclr",
) -> tuple[float, float]:
    """
    Evaluate survival fine-tuning model on a validation or test loader.

    Parameters
    ----------
    model : torch.nn.Module
        Survival model to evaluate.
    data_loader :
        Validation or test loader.
    device : torch.device
        Device on which model and tensors are placed.
    model_type : str, default="simclr"
        Which model interface to use.
        Options:
        - "simclr"
        - "tfc"

    Returns
    -------
    tuple[float, float]
        (average Cox loss, C-index)
    """
    model.eval()

    total_loss = 0.0
    num_batches = 0

    all_risks = []
    all_durations = []
    all_events = []

    with torch.no_grad():
        for batch in data_loader:
            batch = {
                k: v.to(device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }

            if model_type == "simclr":
                _, risk = model(
                    numerical=batch["numerical"],
                    histogram=batch["histogram"],
                    time_gaps=batch["time_gaps"],
                    padding_mask=batch["padding_mask"],
                )

            elif model_type == "tfc":
                x_time = torch.cat(
                    [batch["numerical"], batch["histogram"]],
                    dim=-1,
                )

                _, risk = model(
                    x_time=x_time,
                    time_gaps=batch["time_gaps"],
                    padding_mask=batch["padding_mask"],
                )
            elif model_type == "tfc_static":
                x_time = torch.cat(
                    [batch["numerical"], batch["histogram"]],
                    dim=-1,
                )

                _, risk = model(
                    x_time=x_time,
                    time_gaps=batch["time_gaps"],
                    padding_mask=batch["padding_mask"],
                    static=batch["static_features"],
                )

            else:
                raise ValueError(
                    f"Unknown model_type '{model_type}'. "
                    "Use 'simclr' or 'tfc'."
                )

            loss = cox_partial_log_likelihood(
                risk_scores=risk,
                durations=batch["duration"],
                events=batch["event"],
            )

            total_loss += loss.item()
            num_batches += 1

            all_risks.append(risk.detach().cpu().numpy())
            all_durations.append(batch["duration"].detach().cpu().numpy())
            all_events.append(batch["event"].detach().cpu().numpy())

    avg_loss = total_loss / max(num_batches, 1)

    all_risks = np.concatenate(all_risks, axis=0)
    all_durations = np.concatenate(all_durations, axis=0)
    all_events = np.concatenate(all_events, axis=0)

    c_index = concordance_index_from_risk(
        risk_scores=all_risks,
        durations=all_durations,
        events=all_events,
    )

    return avg_loss, c_index

In [ ]:
import numpy as np
import torch


def train_survival_finetuning(
    model: torch.nn.Module,
    train_loader,
    val_loader,
    device: torch.device,
    model_type: str = "simclr",
    num_epochs: int = 10,
    learning_rate: float = 1e-4,
    weight_decay: float = 0.0,
):
    """
    Train survival fine-tuning model with Cox loss.

    Parameters
    ----------
    model : torch.nn.Module
        Survival fine-tuning model.
    train_loader :
        Training data loader.
    val_loader :
        Validation data loader.
    device : torch.device
        Computation device.
    model_type : str, default="simclr"
        Which model interface to use.
        Options:
        - "simclr"
        - "tfc"
    num_epochs : int, default=10
        Number of epochs.
    learning_rate : float, default=1e-4
        Learning rate.
    weight_decay : float, default=0.0
        Weight decay.

    Returns
    -------
    dict
        Training history with train/val loss and val C-index.
    """
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_c_index": [],
    }

    best_val_c_index = -np.inf
    best_state_dict = None

    for epoch in range(num_epochs):
        model.train()

        total_train_loss = 0.0
        num_train_batches = 0
        skipped_batches = 0

        for batch in train_loader:
            batch = {
                k: v.to(device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }
            batch["static_features"] = batch["static_features"].to(device)

            # Safety check: skip useless batches
            if batch["event"].sum() == 0:
                skipped_batches += 1
                continue

            if model_type == "simclr":
                _, risk = model(
                    numerical=batch["numerical"],
                    histogram=batch["histogram"],
                    time_gaps=batch["time_gaps"],
                    padding_mask=batch["padding_mask"],
                )

            elif model_type in {"tfc", "tfc_static"}:
              x_time = torch.cat(
                  [batch["numerical"], batch["histogram"]],
                  dim=-1,
              )

              if model_type == "tfc_static":
                  _, risk = model(
                      x_time=x_time,
                      time_gaps=batch["time_gaps"],
                      padding_mask=batch["padding_mask"],
                      static=batch["static_features"],
                  )
              else:
                  _, risk = model(
                      x_time=x_time,
                      time_gaps=batch["time_gaps"],
                      padding_mask=batch["padding_mask"],
                  )

            else:
                raise ValueError(
                    f"Unknown model_type '{model_type}'. "
                    "Use 'simclr' or 'tfc', or 'tfc_static'."
                )

            loss = cox_partial_log_likelihood(
                risk_scores=risk,
                durations=batch["duration"],
                events=batch["event"],
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_train_loss += loss.item()
            num_train_batches += 1

        avg_train_loss = total_train_loss / max(num_train_batches, 1)

        val_loss, val_c_index = evaluate_survival_model(
            model=model,
            data_loader=val_loader,
            device=device,
            model_type=model_type,
        )

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(val_loss)
        history["val_c_index"].append(val_c_index)

        if not np.isnan(val_c_index) and val_c_index > best_val_c_index:
            best_val_c_index = val_c_index
            best_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train loss: {avg_train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val C-index: {val_c_index:.4f} | "
            f"Skipped batches: {skipped_batches}"
        )

    return {
        "history": history,
        "best_val_c_index": best_val_c_index,
        "best_state_dict": best_state_dict,
    }

In [ ]:
guided_finetune_results = train_survival_finetuning(
    model=guided_survival_model,
    train_loader=train_surv_loader,
    val_loader=val_surv_loader,
    device=device,
    model_type="tfc",
    num_epochs=40,
    learning_rate=1e-4,
)

In [ ]:
# Check with frozen encoder

guided_base_tfc = TFCSequenceModel(
    time_input_dim=checkpoint["config"]["time_input_dim"],
    freq_input_dim=checkpoint["config"]["freq_input_dim"],
    time_hidden_dim=checkpoint["config"]["time_hidden_dim"],
    freq_hidden_dim=checkpoint["config"]["freq_hidden_dim"],
    projection_dim=checkpoint["config"]["projection_dim"],
).to(device)

guided_base_tfc.load_state_dict(
    guided_results["best_tfc_state_dict"]
)

guided_survival_model_frozen = TFCSurvivalFineTuningModel(
    pretrained_tfc_model=guided_base_tfc,
    survival_hidden_dim=64,
    freeze_encoder=True,
).to(device)

In [ ]:
# fine tune on frozen encoder
guided_finetune_results_frozen = train_survival_finetuning(
    model=guided_survival_model_frozen,
    train_loader=train_surv_loader,
    val_loader=val_surv_loader,
    device=device,
    model_type="tfc",
    num_epochs=40,
    learning_rate=1e-4,
)

## Inclusion of static features for fine tuning on survival downstream task
Tests include direct incorporation of one hot encoded static features, encoding static features through a MLP that compresses them in a 16 or 32 dimensions. Then observe the most stable performance and improved c-index on the validation set.

In [ ]:
# Direct OHE concatenation
model_static_direct = TFCSurvivalFineTuningWithStaticModel(
    pretrained_tfc_model=guided_base_tfc,
    static_dim=89,
    static_fusion="direct",
    survival_hidden_dim=64,
    dropout=0.2,
).to(device)

In [ ]:
# MLP OHE representation repr_dim = 16
model_static_mlp = TFCSurvivalFineTuningWithStaticModel(
    pretrained_tfc_model=guided_base_tfc,
    static_dim=89,
    static_fusion="mlp",
    static_repr_dim=16,
    survival_hidden_dim=64,
    dropout=0.2,
).to(device)

In [ ]:
# MLP OHE representation repr_dim = 32
model_static_mlp32 = TFCSurvivalFineTuningWithStaticModel(
    pretrained_tfc_model=guided_base_tfc,
    static_dim=89,
    static_fusion="mlp",
    static_repr_dim=32,
    survival_hidden_dim=64,
    dropout=0.2,
).to(device)

In [ ]:
# training with direct ohe
results_static_direct = train_survival_finetuning(
    model=model_static_direct,
    train_loader=train_surv_loader,
    val_loader=val_surv_loader,
    device=device,
    model_type="tfc_static",
    num_epochs=40,
    learning_rate=1e-4,
    weight_decay=1e-4,
)

In [ ]:
# training with mlp-compressed from 89 to 16 dimensions
results_static_mlp = train_survival_finetuning(
    model=model_static_mlp,
    train_loader=train_surv_loader,
    val_loader=val_surv_loader,
    device=device,
    model_type="tfc_static",
    num_epochs=40,
    learning_rate=1e-4,
    weight_decay=1e-4,
)

In [ ]:
# training with mlp-compressed from 89 to 32 dimensions
results_static_mlp32 = train_survival_finetuning(
    model=model_static_mlp32,
    train_loader=train_surv_loader,
    val_loader=val_surv_loader,
    device=device,
    model_type="tfc_static",
    num_epochs=40,
    learning_rate=1e-4,
    weight_decay=1e-4,
)

In [ ]:
# plot results
import matplotlib.pyplot as plt


def plot_survival_finetuning_results(results_dict, save_dir=None):
    """
    Plot training loss, validation loss, and validation C-index
    for multiple survival fine-tuning models.
    """

    plots = [
        ("train_loss", "Training loss", "Training Loss During Fine-tuning"),
        ("val_loss", "Validation loss", "Validation Loss During Fine-tuning"),
        ("val_c_index", "Validation C-index", "Validation C-index During Fine-tuning"),
    ]

    for metric_key, ylabel, title in plots:
        plt.figure(figsize=(8, 5))

        for model_name, result in results_dict.items():
            history = result["history"]
            values = history[metric_key]
            epochs = range(1, len(values) + 1)

            plt.plot(
                epochs,
                values,
                linewidth=2,
                label=model_name.replace("_", " "),
            )

        plt.xlabel("Epoch", fontsize=11)
        plt.ylabel(ylabel, fontsize=11)
        plt.title(title, fontsize=13)
        plt.legend(frameon=False, fontsize=9)
        plt.tight_layout()

        if save_dir is not None:
            filename = metric_key + ".png"
            plt.savefig(
                f"{save_dir}/{filename}",
                dpi=300,
                bbox_inches="tight",
            )

        plt.show()

In [ ]:
results_dict = {
    "guided_ft_tfc_frozen": guided_finetune_results_frozen,
    "guided_ft_tfc": guided_finetune_results,
    "tfc_static_direct": results_static_direct,
    "tfc_static_mlp16": results_static_mlp,
    "tfc_static_mlp32": results_static_mlp32,
}

plot_survival_finetuning_results(results_dict)

In [ ]:
# save the best model for the selected approach
save_path_best_ssl = MODELS_DIR / "best_tfc_static_mlp32_finetuned.pt"

torch.save(
    {
        "best_state_dict": results_static_mlp32["best_state_dict"],
        "history": results_static_mlp32["history"],
        "best_val_c_index": results_static_mlp32["best_val_c_index"],
        "model_name": "tfc_static_mlp32",
        "config": {
            "time_input_dim": 105,
            "static_dim": 89,
            "time_hidden_dim": 128,
            "survival_hidden_dim": 64,
            "static_fusion": "mlp",
            "dropout": 0.2,
            "freeze_encoder": False,
        },
    },
    save_path_best_ssl,
)

print("Saved best SSL model to:", save_path_best_ssl)

In [ ]:
# evaluate on test set for final results
test_results = {}

models_to_test = {
    "guided_ft_tfc_frozen": (guided_survival_model_frozen, guided_finetune_results_frozen),
    "guided_ft_tfc": (guided_survival_model, guided_finetune_results),
    "tfc_static_direct": (model_static_direct, results_static_direct),
    "tfc_static_mlp16": (model_static_mlp, results_static_mlp),
    "tfc_static_mlp32": (model_static_mlp32, results_static_mlp32),
}

for model_name, (model, result) in models_to_test.items():
    model.load_state_dict(result["best_state_dict"])
    model.to(device)
    model.eval()

    model_type = "tfc_static" if "static" in model_name else "tfc"

    test_loss, test_c_index = evaluate_survival_model(
        model=model,
        data_loader=test_surv_loader,
        device=device,
        model_type=model_type,
    )

    test_results[model_name] = {
        "best_val_c_index": result["best_val_c_index"],
        "test_loss": test_loss,
        "test_c_index": test_c_index,
    }

test_results

In [ ]:
# save results
import json
import pandas as pd

test_results_df = pd.DataFrame.from_dict(test_results, orient="index")
test_results_df = test_results_df.sort_values("test_c_index", ascending=False)

save_csv = "/content/drive/MyDrive/SSL_SCANIA_CompX/results/ssl_test_results.csv"
save_json = "/content/drive/MyDrive/SSL_SCANIA_CompX/results/ssl_test_results.json"

test_results_df.to_csv(save_csv)

with open(save_json, "w") as f:
    json.dump(test_results, f, indent=4)

print(test_results_df)
print("Saved CSV:", save_csv)
print("Saved JSON:", save_json)

# More evaluation metrics
After final results of the proposed approach on the test set, the best model will now be evaluated on more evaluation metrics such as the Integrated Brier Score (IBS) and the time-dependent AUC. For that to be done, survival probabilities must be computed using the Breslow estimator. This will be obtained from the training set, using the following set of functions.

In [ ]:
# collect risk scores, durations and event indicators from the model
import numpy as np
import torch
from sksurv.linear_model.coxph import BreslowEstimator
from sksurv.metrics import cumulative_dynamic_auc, integrated_brier_score


def make_structured_survival_array(events, durations):
    """
    Convert event/duration arrays to scikit-survival structured format.
    """
    return np.array(
        [(bool(e), float(t)) for e, t in zip(events, durations)],
        dtype=[("event", "?"), ("time", "<f8")],
    )


def collect_ssl_risk_scores_and_targets(
    model,
    data_loader,
    device,
    model_type="tfc_static",
):
    """
    Collect risk scores, durations, and event indicators from an SSL Cox-style model.
    """

    model.eval()

    all_risk = []
    all_duration = []
    all_event = []

    with torch.no_grad():
        for batch in data_loader:
            batch = {
                k: v.to(device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }

            if model_type in {"tfc", "tfc_static"}:
                x_time = torch.cat(
                    [batch["numerical"], batch["histogram"]],
                    dim=-1,
                )

                if model_type == "tfc_static":
                    _, risk = model(
                        x_time=x_time,
                        time_gaps=batch["time_gaps"],
                        padding_mask=batch["padding_mask"],
                        static=batch["static_features"].float(),
                    )
                else:
                    _, risk = model(
                        x_time=x_time,
                        time_gaps=batch["time_gaps"],
                        padding_mask=batch["padding_mask"],
                    )

            else:
                raise ValueError("Use model_type='tfc' or 'tfc_static'.")

            all_risk.append(risk.detach().cpu().numpy())
            all_duration.append(batch["duration"].detach().cpu().numpy())
            all_event.append(batch["event"].detach().cpu().numpy())

    risk_scores = np.concatenate(all_risk)
    durations = np.concatenate(all_duration)
    events = np.concatenate(all_event).astype(bool)

    return risk_scores, durations, events

In [ ]:
# main metric function
def evaluate_ssl_survival_curves_with_breslow(
    model,
    train_loader,
    test_loader,
    device,
    model_type="tfc_static",
    n_time_points=100,
):
    """
    Evaluate SSL Cox-style model using:
    - risk scores
    - Breslow-estimated survival curves
    - time-dependent AUC
    - integrated Brier score

    Returns
    -------
    dict
        Dictionary containing mean AUC, AUC values, IBS, time grid,
        survival probability matrix, and risk scores.
    """

    # 1. Collect train predictions and targets
    train_risk, train_durations, train_events = collect_ssl_risk_scores_and_targets(
        model=model,
        data_loader=train_loader,
        device=device,
        model_type=model_type,
    )

    # 2. Collect test predictions and targets
    test_risk, test_durations, test_events = collect_ssl_risk_scores_and_targets(
        model=model,
        data_loader=test_loader,
        device=device,
        model_type=model_type,
    )

    # 3. Convert to sksurv structured arrays
    y_train = make_structured_survival_array(
        events=train_events,
        durations=train_durations,
    )

    y_test = make_structured_survival_array(
        events=test_events,
        durations=test_durations,
    )

    # 4. Fit Breslow baseline hazard on training set
    breslow = BreslowEstimator()
    breslow.fit(
        linear_predictor=train_risk,
        event=train_events,
        time=train_durations,
    )

    # 5. Define evaluation time grid
    min_time = np.percentile(test_durations, 5)
    max_time = np.percentile(test_durations, 95)

    times = np.linspace(min_time, max_time, n_time_points)

    # Ensure times are within training support
    times = times[
        (times > train_durations.min()) &
        (times < train_durations.max())
    ]

    # 6. Estimate survival functions for test samples
    surv_funcs = breslow.get_survival_function(test_risk)

    surv_matrix = np.vstack([
        fn(times) for fn in surv_funcs
    ])

    # 7. Time-dependent AUC
    auc_values, mean_auc = cumulative_dynamic_auc(
        survival_train=y_train,
        survival_test=y_test,
        estimate=test_risk,
        times=times,
    )

    # 8. Integrated Brier Score
    ibs = integrated_brier_score(
        survival_train=y_train,
        survival_test=y_test,
        estimate=surv_matrix,
        times=times,
    )

    return {
        "mean_auc": mean_auc,
        "auc_values": auc_values,
        "ibs": ibs,
        "times": times,
        "surv_matrix": surv_matrix,
        "test_risk": test_risk,
        "test_durations": test_durations,
        "test_events": test_events,
    }

In [ ]:
# apply metric on the model and dataset
# Load best state first
checkpoint = torch.load(
    MODELS_DIR / "best_tfc_static_mlp32_finetuned.pt",
    map_location=device,
    weights_only=False,
)
config = checkpoint["config"]

base_tfc = TFCSequenceModel(
    time_input_dim=config["time_input_dim"],
    freq_input_dim=config.get("freq_input_dim", 105),
    time_hidden_dim=config["time_hidden_dim"],
    freq_hidden_dim=config.get("freq_hidden_dim", 128),
    projection_dim=config.get("projection_dim", 64),
).to(device)

model_static_mlp32 = TFCSurvivalFineTuningWithStaticModel(
    pretrained_tfc_model=base_tfc,
    static_dim=config["static_dim"],
    static_fusion=config["static_fusion"],
    static_hidden_dim=config.get("static_hidden_dim", 32),
    static_repr_dim=config.get("static_repr_dim", 32),
    survival_hidden_dim=config["survival_hidden_dim"],
    dropout=config.get("dropout", 0.2),
    freeze_encoder=False,
).to(device)

model_static_mlp32.load_state_dict(checkpoint["best_state_dict"])
model_static_mlp32.eval()

In [ ]:
ssl_curve_metrics = evaluate_ssl_survival_curves_with_breslow(
    model=model_static_mlp32,
    train_loader=train_surv_loader,
    test_loader=test_surv_loader,
    device=device,
    model_type="tfc_static",
    n_time_points=100,
)

print("Mean time-dependent AUC:", ssl_curve_metrics["mean_auc"])
print("Integrated Brier Score:", ssl_curve_metrics["ibs"])

In [ ]:
# save the resulting metrics

import json
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(project_root)

RESULTS_DIR = PROJECT_ROOT / "results" / "final"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ssl_metrics_summary = {
    "model": "SSL TFC static MLP32",
    "c_index": 0.728424,
    "mean_auc": float(ssl_curve_metrics["mean_auc"]),
    "ibs": float(ssl_curve_metrics["ibs"]),
}

with open(RESULTS_DIR / "ssl_curve_metrics_summary.json", "w") as f:
    json.dump(ssl_metrics_summary, f, indent=4)

pd.DataFrame({
    "time": ssl_curve_metrics["times"],
    "auc": ssl_curve_metrics["auc_values"],
}).to_csv(RESULTS_DIR / "ssl_time_dependent_auc.csv", index=False)

np.save(RESULTS_DIR / "ssl_survival_times.npy", ssl_curve_metrics["times"])
np.save(RESULTS_DIR / "ssl_survival_matrix.npy", ssl_curve_metrics["surv_matrix"])
np.save(RESULTS_DIR / "ssl_test_risk.npy", ssl_curve_metrics["test_risk"])

## Towards building the final comparison table
The following section presents the evaluation metrics for all models considered in this study. For both the proposed semi-supervised learning (SSL) approach and the Random Survival Forest (RSF) model, all evaluation metrics, namely the concordance index (C-index), time-dependent Area Under the Curve (AUC), and Integrated Brier Score (IBS)—were successfully computed.

The same evaluation protocol was applied to the Cox Proportional Hazards (Cox PH) model. However, while valid C-index and time-dependent AUC values were obtained, the model produced numerically unstable survival probability estimates. This instability is attributable to extremely large linear predictor values, which resulted in degenerate survival curves. Consequently, the Integrated Brier Score (IBS) is not reported for the Cox PH model, as it would not provide a meaningful assessment of its probabilistic performance.

In [ ]:
# ssl metrics
ssl_c_index = 0.728424
ssl_mean_auc = 0.7901162245446707
ssl_ibs = 0.07830150265411198
# rsf metrics
rsf_c_index = 0.716737
rsf_mean_auc = 0.754738
rsf_ibs = 0.091107
# cox metrics
cph_c_index = 0.658040
cph_mean_auc = 0.704465
# no IBS

In [ ]:
# final comparison table
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(project_root) / "results" / "final"

final_table = pd.DataFrame([
    {
        "Model": "Proposed approach",
        "C-index": ssl_c_index,
        "Mean AUC": ssl_mean_auc,
        "IBS": ssl_ibs,
    },
    {
        "Model": "RSF (with static)",
        "C-index": rsf_c_index,
        "Mean AUC": rsf_mean_auc,
        "IBS": rsf_ibs,
    },
    {
        "Model": "Cox PH",
        "C-index": cph_c_index,
        "Mean AUC": cph_mean_auc,
        "IBS": None,
    },
])

# formatting for presentation
final_table["C-index"] = final_table["C-index"].map(lambda x: f"{x:.3f}")
final_table["Mean AUC"] = final_table["Mean AUC"].map(lambda x: f"{x:.3f}")
final_table["IBS"] = final_table["IBS"].map(lambda x: f"{x:.3f}" if pd.notnull(x) else "—")

print(final_table)

In [ ]:
# save results
final_table.to_csv(RESULTS_DIR / "final_results_table.csv", index=False)

with open(RESULTS_DIR / "final_results_table.tex", "w") as f:
    f.write(final_table.to_latex(index=False))

In [ ]:
# plot a number of survival curves for selected vehicles, 2 for each risk category (low, medium, high)
import numpy as np
import matplotlib.pyplot as plt


def plot_representative_ssl_survival_curves(ssl_curve_metrics):
    times = ssl_curve_metrics["times"]
    surv_matrix = ssl_curve_metrics["surv_matrix"]
    risk_scores = ssl_curve_metrics["test_risk"]

    selected = {
        "Low risk (10th percentile)": np.argmin(
            np.abs(risk_scores - np.percentile(risk_scores, 10))
        ),
        "Median risk (50th percentile)": np.argmin(
            np.abs(risk_scores - np.percentile(risk_scores, 50))
        ),
        "High risk (90th percentile)": np.argmin(
            np.abs(risk_scores - np.percentile(risk_scores, 90))
        ),
    }

    plt.figure(figsize=(8, 5))

    for label, idx in selected.items():
        plt.step(
            times,
            surv_matrix[idx],
            where="post",
            linewidth=2,
            label=label,
        )

    plt.xlabel("Time")
    plt.ylabel("Predicted survival probability")
    plt.title("Representative Predicted Survival Curves")
    plt.ylim(0, 1.02)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_representative_ssl_survival_curves(
    ssl_curve_metrics=ssl_curve_metrics
)